In [23]:
!pip install faiss-cpu

In [24]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [35]:
documents = [

"""
Diabetes symptoms include increased thirst, fatigue,
frequent urination, blurred vision, and weight loss.
Patients should monitor blood glucose regularly.
""",

"""
Chest pain accompanied by shortness of breath,
sweating, nausea, or dizziness may indicate
a cardiac emergency requiring immediate medical care.
""",

"""
Dehydration symptoms include dizziness, dry mouth,
low urine output, weakness, and confusion.
Hydration is essential for recovery.
""",

"""
Hospital readmission risk increases in elderly patients,
patients with chronic illness, and patients with
multiple prior admissions.
""",

"""
High blood pressure may increase the risk of heart disease,
stroke, and kidney complications if untreated.
"""
]

In [36]:
embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
document_embeddings = embedding_model.encode(
    documents
)

In [38]:
dimension = document_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(document_embeddings)
)

print("Vector DB Ready")

Vector DB Ready


In [39]:
def retrieve_documents(query, top_k=2):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append(documents[idx])

    return results

In [40]:
retrieve_documents(
    "What are symptoms of diabetes?"
)

['\nDiabetes symptoms include increased thirst, fatigue,\nfrequent urination, blurred vision, and weight loss.\nPatients should monitor blood glucose regularly.\n',
 '\nDehydration symptoms include dizziness, dry mouth,\nlow urine output, weakness, and confusion.\nHydration is essential for recovery.\n']

In [41]:
def summary_agent(question):

    return f"""
Patient Summary:
- Concern: {question}
- AI-generated preliminary healthcare guidance created
- Doctor consultation recommended if symptoms continue
"""

In [42]:
def coordinator_agent(question):

    final_response = ""

    # Emergency check

    emergency_response = emergency_agent(question)

    if emergency_response:

        final_response += emergency_response + "\n"

    # Risk analysis

    risk_response = risk_agent(question)

    if risk_response:

        final_response += risk_response + "\n"

    # Retrieval system

    retrieval_response = retrieval_agent(question)

    final_response += retrieval_response + "\n"

    # Summary

    summary_response = summary_agent(question)

    final_response += summary_response

    return final_response

In [ ]:
while True:

    question = input("Patient: ")

    if question.lower() == "exit":

        print("MediSphere AI: Stay healthy.")
        break

    response = coordinator_agent(question)

    print("\nMediSphere AI:")
    print(response)
    print("\n")

Patient: What are symptoms of dehydration?

MediSphere AI:

Retrieved Medical Knowledge:


Dehydration symptoms include dizziness, dry mouth,
low urine output, weakness, and confusion.
Hydration is essential for recovery.


Diabetes symptoms include increased thirst, fatigue,
frequent urination, blurred vision, and weight loss.
Patients should monitor blood glucose regularly.



Patient Summary:
- Concern: What are symptoms of dehydration?
- AI-generated preliminary healthcare guidance created
- Doctor consultation recommended if symptoms continue



Patient: I have diabetes and dizziness

MediSphere AI:

Risk Agent Analysis:
Patient may belong to a higher-risk clinical category.
Recommend monitoring and physician consultation.


Retrieved Medical Knowledge:


Diabetes symptoms include increased thirst, fatigue,
frequent urination, blurred vision, and weight loss.
Patients should monitor blood glucose regularly.


Dehydration symptoms include dizziness, dry mouth,
low urine output, wea